# Text Classification (Sentiment)

## Setup

### Import Dependencies

In [1]:
import pandas as pd
import os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

C:\Users\ethan\stock_price_prediction2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load CSV file into DataFrame

In [2]:
load_dotenv()

df = pd.read_csv("../data/transformed_news_data.csv")

df.head()

,id,published_date,title,source,tags,crawl_date,tickers,day_of_week
0,95549488,2026-05-12,New lawsuit claims Facebook profits by 'active...,nypost.com,"('California', 'ETF', 'Lawsuits', 'Meta', 'Sca...",2026-05-12,"('goog', 'googl', 'meta', 'meta', 'metv')",2
1,95540140,2026-05-12,Google races to put Gemini at the center of An...,cnbc.com,"('Breaking News: Technology', 'Business', 'Com...",2026-05-12,"('aapl', 'adbe', 'cart', 'goog', 'googl', 'igv...",2
2,95535302,2026-05-12,Laid off GM employees describe ominous meeting...,cnbc.com,"('Autos', 'Breaking News: Business', 'Business...",2026-05-12,"('amzn', 'meta', 'meta', 'metv', 'orcl', 'xyz')",2
3,95533695,2026-05-12,EU Announces Plans To Ban Addictive Social Med...,ibtimes.com,"('Conflict', 'ETF', 'Eu', 'Eu Banning Social M...",2026-05-12,"('eu', 'meta', 'meta', 'metv')",2
4,95532956,2026-05-12,Google Unveils New Android AI FeaturesAhead of...,bloomberg.com,"('Stock', 'Technology', 'Tiingo Top')",2026-05-12,"('aapl', 'goog', 'googl', 'meta')",2


## Text Classification (sentiment)

### Run model on news data

In [ ]:
def run_text_classification(df):
    client = InferenceClient(
        provider="hf-inference",
        api_key=os.getenv('hugging_face_token'),
    )

    results = client.text_classification(
        df['title'].to_list(),
        model="ProsusAI/finbert",
    )

    # Generated with AI (Gemini 3 fast)
    df['sentiment_label'] = [res.label for res in results]
    df['sentiment_score'] = [res.score for res in results]
    return df

df = run_text_classification(df)
df.head()

In [9]:
def enrich_df(df):
    # Assign numerical labels to each sentiment
    df['numerical_sentiment'] = df.apply(lambda x: \
                                         1 if x['sentiment_label'] == 'positive'
                                         else (0 if x['sentiment_label'] == 'neutral' else -1),
                                         axis=1)

    # Flag if an article is positive, negative or neutral
    df['is_positive'] = df.apply(lambda x: 1 if x['sentiment_label'] == 'positive' else 0, axis=1)
    df['is_negative'] = df.apply(lambda x: 1 if x['sentiment_label'] == 'negative' else 0, axis=1)
    df['is_neutral'] = df.apply(lambda x: 1 if x['sentiment_label'] == 'neutral' else 0, axis=1)

    return df

df = enrich_df(df)
df.head()

,id,published_date,title,source,tags,crawl_date,tickers,day_of_week,sentiment_label,sentiment_score,numerical_sentiment,is_positive,is_negative,is_neutral
0,94624618,2026-04-19,Prediction: It's Not Too Late to Buy Broadcom ...,finance.yahoo.com,"['Broadcom', 'Broadcom Stock', 'Communication ...",2026-04-19,"['avgo', 'brcm', 'intc', 'iqnt', 'meta', 'nflx...",7,neutral,0.810057,0,0,0,1
1,94622595,2026-04-19,This Stock Will Be More Profitable Than Amazon...,finance.yahoo.com,"['Amazon', 'Communication Services', 'Compound...",2026-04-19,"['amd', 'amzn', 'intc', 'iqnt', 'meta', 'mu', ...",7,positive,0.585464,1,1,0,0
2,94621400,2026-04-19,Option Traders Chasing Torrid Stock Rally Turn...,bloomberg.com,"['Financial Services', 'Stock', 'Technology', ...",2026-04-19,"['jpm', 'meta', 'msft']",7,negative,0.485695,-1,0,1,0
3,94612855,2026-04-18,Broadcom (AVGO) Ranks Among Unrivaled Stocks o...,finance.yahoo.com,"['Blue Chip Stocks', 'Broadcom Inc.', 'Inc.', ...",2026-04-18,"['avgo', 'brcm', 'meta']",6,positive,0.770807,1,1,0,0
4,94612626,2026-04-18,Meta Platforms (META) Expands AI Chip Deal wit...,insidermonkey.com,"['Stock', 'Technology']",2026-04-18,['meta'],6,positive,0.745573,1,1,0,0


### Calculate Metrics

In [10]:
def calculate_metrics(df):
    # Calculate metrics
    metrics_df = df.groupby(['published_date'])\
                    .agg({
                    'numerical_sentiment': 'mean',
                    'sentiment_score': 'mean',
                    'is_positive': 'sum',
                    'is_negative': 'sum',
                    'is_neutral': 'sum',
                    'sentiment_label': 'count'})\
                    .reset_index()\
                    .rename(
                    columns={'published_date': 'date',
                             'is_positive': 'positive_count',
                             'is_negative': 'negative_count',
                             'is_neutral': 'neutral_count',
                             'sentiment_label': 'total_articles',
                             'numerical': 'mean_sentiment',
                             'sentiment_score': 'mean_sentiment_probability'})

    metrics_df.to_csv('data/sentiment_counts.csv', mode='a', index=False)
    sentiment_counts_df = pd.read_csv('data/sentiment_counts.csv').drop_duplicates(subset=['date'], keep='first')
    sentiment_counts_df.to_csv('data/sentiment_counts.csv', index=False)

    # Calculate Percentages
    metrics_df['percent_positive'] = metrics_df['positive_count'] / metrics_df['total_articles']
    metrics_df['percent_negative'] = metrics_df['negative_count'] / metrics_df['total_articles']
    metrics_df['percent_neutral'] = metrics_df['neutral_count'] / metrics_df['total_articles']

    # Remove count columns use the percentages of totals instead
    metrics_df = metrics_df.drop(['positive_count', 'negative_count', 'neutral_count', 'total_articles'], axis=1)

    return metrics_df

metrics_df = calculate_metrics(df)
metrics_df.head()

,date,numerical_sentiment,mean_sentiment_probability,percent_positive,percent_negative,percent_neutral
0,2026-04-14,0.076923,0.794120,0.230769,0.153846,0.615385
1,2026-04-15,0.000000,0.786592,0.242424,0.242424,0.515152
2,2026-04-16,0.333333,0.783817,0.333333,0.000000,0.666667
3,2026-04-17,0.062500,0.786713,0.187500,0.125000,0.687500
4,2026-04-18,0.090909,0.824430,0.272727,0.181818,0.545455


## Write output DataFrame as CSV File

In [11]:
# Output as CSV
metrics_df.to_csv('../data/article_metrics.csv', index=False)